# Lab 01 — Image Processing Fundamentals, Spatial Filtering & 2D FFT

**Course:** Image Analysis and Processing — École des Mines de Saint-Étienne  
**Topics covered:**
1. Manual histogram computation and validation against `numpy.histogram`
2. Linear contrast stretching (min-max intensity normalization)
3. Spatial low-pass & rank filtering (mean, median, min, max, Gaussian)
4. 2D Discrete Fourier Transform (amplitude and phase spectra)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage import filters, morphology
from skimage.filters import rank

IMAGES_DIR = Path("../Images")

## 1. Custom Histogram Implementation

We implement a histogram function from scratch using `np.bincount` and validate that both bin counts and bin edges match `np.histogram`.

In [ ]:
def compute_histogram(image: np.ndarray, bins: int = 256):
    """Compute the intensity histogram of an image from scratch."""
    values = np.asarray(image).ravel()
    data_min, data_max = values.min(), values.max()
    edges = np.linspace(data_min, data_max, bins + 1)

    if data_max == data_min:
        counts = np.zeros(bins, dtype=int)
        counts[0] = values.size
        return counts, edges

    scaled = (values - data_min) / (data_max - data_min) * bins
    indices = np.clip(np.floor(scaled).astype(int), 0, bins - 1)
    counts = np.bincount(indices, minlength=bins)
    return counts, edges


muscle = imread(IMAGES_DIR / "muscle.jpg")
h_manual, edges_manual = compute_histogram(muscle, bins=256)
h_numpy, edges_numpy = np.histogram(muscle, bins=256)

print("Counts match NumPy:", np.array_equal(h_manual, h_numpy))
print("Edges match NumPy: ", np.allclose(edges_manual, edges_numpy))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(muscle, cmap="gray")
axes[0].set_title("Muscle Tissue (Original)")
axes[0].axis("off")

axes[1].plot(edges_manual[:-1], h_manual, color="navy")
axes[1].set_title("Intensity Histogram")
axes[1].set_xlabel("Pixel Intensity")
axes[1].set_ylabel("Count")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Linear Contrast Stretching

We expand the dynamic range of a low-contrast corneal cell image to fill the full `[0, 255]` range using affine rescaling:

$$I_{\text{stretched}} = \frac{255}{I_{\max} - I_{\min}} (I - I_{\min})$$

In [ ]:
cornea = imread(IMAGES_DIR / "cellules_cornee.jpg").astype(np.float64)

v_min, v_max = cornea.min(), cornea.max()
slope = 255.0 / (v_max - v_min)
intercept = -slope * v_min
cornea_stretched = slope * cornea + intercept

print(f"Original range:  [{v_min:.1f}, {v_max:.1f}]")
print(f"Stretched range: [{cornea_stretched.min():.1f}, {cornea_stretched.max():.1f}]")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cornea, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original (Low Contrast)")
axes[0].axis("off")

axes[1].imshow(cornea_stretched, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("After Linear Contrast Stretching")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 3. Spatial Low-Pass & Rank Filtering

We compare five spatial filters (Mean, Median, Minimum, Maximum, and Gaussian) on a microscopy image of blood cells using a disk structuring element of radius 3.

In [ ]:
blood = imread(IMAGES_DIR / "blood.jpg")
print(f"Image shape: {blood.shape}, dtype: {blood.dtype}")

footprint = morphology.disk(3)
blood_mean = rank.mean(blood, footprint)
blood_median = rank.median(blood, footprint)
blood_min = rank.minimum(blood, footprint)
blood_max = rank.maximum(blood, footprint)
blood_gaussian = filters.gaussian(blood, sigma=1.0)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
panels = [
    (blood, "Original", (0, 255)),
    (blood_mean, "Mean Filter (disk r=3)", (0, 255)),
    (blood_median, "Median Filter (disk r=3)", (0, 255)),
    (blood_min, "Minimum Filter (Erosion)", (0, 255)),
    (blood_max, "Maximum Filter (Dilation)", (0, 255)),
    (blood_gaussian, "Gaussian Filter (sigma=1)", (0, 1)),
]
for ax, (img, title, (vmin, vmax)) in zip(axes.flat, panels):
    ax.imshow(img, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. 2D Discrete Fourier Transform

We compute the centered 2D FFT of a retinal image and visualize its log-scaled amplitude spectrum alongside its phase spectrum.

In [ ]:
retina = imread(IMAGES_DIR / "retine.png", as_gray=True).astype(np.float64)
if retina.max() > 1.0:
    retina /= 255.0

fft_shifted = np.fft.fftshift(np.fft.fft2(retina))
amplitude = np.abs(fft_shifted)
phase = np.angle(fft_shifted)

amplitude_log = np.log1p(amplitude)
amplitude_display = (amplitude_log - amplitude_log.min()) / (
    amplitude_log.max() - amplitude_log.min()
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(retina, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original Retina Image")
axes[0].axis("off")

axes[1].imshow(amplitude_display, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Amplitude Spectrum (Log Scale)")
axes[1].axis("off")

axes[2].imshow(phase, cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[2].set_title("Phase Spectrum")
axes[2].axis("off")

plt.tight_layout()
plt.show()